In [108]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

from networkx.algorithms import community
from networkx.algorithms.community import greedy_modularity_communities
from operator import itemgetter
import csv


In [5]:
cd /Users/thb5018/Downloads/

/Users/thb5018/Downloads


In [29]:
country = 'iran'

df = pd.read_csv('{}_edgelist.csv'.format(country))
df2 = pd.read_csv('{}_edges.csv'.format(country))
df3 = pd.merge(df,df2[['Target','country']].drop_duplicates(),on='Target', how='inner')
snodes = df3[['source']].copy()
snodes['country'] = country
snodes = snodes.rename(columns={'source': 'node'})
snodes = snodes.drop_duplicates()

tnodes = df3[['Target', 'country']].copy()
tnodes = tnodes.rename(columns={'Target': 'node'})
tnodes = tnodes.drop_duplicates()

nodes = pd.concat([snodes,tnodes])
df_attributes_only = nodes.drop_duplicates()
node_attr = df_attributes_only.set_index('node').to_dict('index')

# Create nodelist

df_attributes_only['codes']=pd.Categorical(df_attributes_only['country'].cat.codes)
print(df_attributes_only)

df_attributes_only.to_csv('{}_nodelist.csv'.format(country), index=False)

# Create Edgelist
df = df.rename(columns={'source':'Source'})
edges = df[['Source','Target','weight']]
edges.to_csv('{}_edgelist.csv'.format(country), index=False)

In [97]:
with open('{}_nodelist.csv'.format(country), 'r') as nodecsv: # Open the file
    nodereader = csv.reader(nodecsv) # Read the csv
    # Retrieve the data (using Python list comprhension and list slicing to remove the header row, see footnote 3)
    nodes = [n for n in nodereader][1:]

node_names = [n[0] for n in nodes] # Get a list of only the node names

with open('{}_edgelist.csv'.format(country), 'r') as edgecsv: # Open the file
    edgereader = csv.reader(edgecsv) # Read the csv
    edges = [tuple(e) for e in edgereader][1:] # Retrieve the data


# Create Graph

In [98]:
G = nx.Graph()


G.add_nodes_from(node_names)
G.add_weighted_edges_from(edges)


print(nx.info(G))

# Create an empty dictionary for each attribute
country_dict = {}
country_code_dict = {} 

for node in nodes: # Loop through the list of nodes, one row at a time
    country_dict[node[0]] = node[1] # Access the correct item, add it to the corresponding dictionary
    country_code_dict[node[0]] = node[2]

# Add each dictionary as a node attribute to the Graph object
nx.set_node_attributes(G, country_dict, 'country')
nx.set_node_attributes(G, country_code_dict, 'code')

# Loop through each node, to access and print all the "country" attributes
for n in G.nodes():
    print(n, G.nodes[n]['country'])

Name: 
Type: Graph
Number of nodes: 603
Number of edges: 801
Average degree:   2.6567


+AL7riKucmp7iF1r1UxC5KytaCNzV1Y+ykf83lDdw= iran
0M6q78TgoPkgXsQlLdCkIbNgW1tTqbN6Xe1234IaQ= iran
2c538df357dec4c1413851b5f1b68da2d7d1f6df4157be85d15901b0913ff714 iran
5eb2c59301da5d5249e7c40e1c4abe22dfd9e9e2a3b6ce60fa9bb080b673e61c iran
719e0500827a4fdcf610520054f04d4ec13c7125b83c0a23fc6c75ca86a8cf6b iran
7AihqeEJRHKt30Sl81r6lQenHQc8IR1yGv6Yc+yzTU= iran
7H7hry65IojQY4jTQFoUPhBu6voB6Qz6lJ38+oNHOrE= iran
7tvS6RtKDHvcHubX4qpH0gwH0DFwVpBhMFu9EwkyBM= iran
95RWp7S3oRpL0XfKu3kymI0Li9Q74BukxvgbpdW0Gg= iran
9PZgPUKeb14sqtKQg0Gv1RQj2+BDAcgbytJ6Q7f8f54= iran
Abwu6+0+JS0EiagVTAOQMIzYJbxbR3tnRJ5Y3qjR8c= iran
aGj8ZHdbONkeiIdk5MZ10Llz3JqjsuBbZHfAp8riMyU= iran
Au2wwfJNPHStbKAqBJlcFz3RfmhCjiUuXnEyFtrdUL4= iran
bgUEpUr07MmsAhTLcaWA17YIyOW3msbYARWNBQJKlIs= iran
BujoF63frVdoC2d4rM0skMPX1ateC82jo77wPYf22w= iran
cdPMJc+fn48chvYdO2qk+0HnpcTJ6MWNU2jHlfg9qLE= iran
CPJMkifNgcvn02KidhWvaB+QD4yHaD7z7Z2YkcUJzhI= iran
dEy+qR7R+4iN4BBPl0KVz938J9VLNL5qCN68qdMgOWo= iran
duKXkVVrHhAoyIdbMQAVjQDw7OLcAx3tKFX8imqBQQ= iran


# Metrics

In [122]:
density = nx.density(G)
print("Network density:", density)

# If your Graph has more than one component, this will return False:
print(nx.is_connected(G))

# Next, use nx.connected_components to get the list of components,
# then use the max() command to find the largest one:
components = nx.connected_components(G)
largest_component = max(components, key=len)

# Create a "subgraph" of just the largest component
# Then calculate the diameter of the subgraph, just like you did with density.
#

subgraph = G.subgraph(largest_component)
diameter = nx.diameter(subgraph)
print("Network diameter of largest component:", diameter)

triadic_closure = nx.transitivity(G)
print("Triadic closure:", triadic_closure)

degree_dict = dict(G.degree(G.nodes()))
nx.set_node_attributes(G, degree_dict, 'degree')

sorted_degree = sorted(degree_dict.items(), key=itemgetter(1), reverse=True)

print("Top 20 nodes by degree:")
for d in sorted_degree[:20]:
    print(d)
    
betweenness_dict = nx.betweenness_centrality(G) # Run betweenness centrality
eigenvector_dict = nx.eigenvector_centrality(G) # Run eigenvector centrality

# Assign each to an attribute in your network
nx.set_node_attributes(G, betweenness_dict, 'betweenness')
nx.set_node_attributes(G, eigenvector_dict, 'eigenvector')

sorted_betweenness = sorted(betweenness_dict.items(), key=itemgetter(1), reverse=True)

print("Top 20 nodes by betweenness centrality:")
for b in sorted_betweenness[:20]:
    print(b)

#First get the top 20 nodes by betweenness as a list
top_betweenness = sorted_betweenness[:20]

#Then find and print their degree
for tb in top_betweenness: # Loop through top_betweenness
    degree = degree_dict[tb[0]] # Use degree_dict to access a node's degree, see footnote 2
    print("Name:", tb[0], "| Betweenness Centrality:", tb[1], "| Degree:", degree)


for u,v,d in G.edges(data=True):
    d['weight'] = float(d['weight'])
communities = greedy_modularity_communities(G)

modularity_dict = {} # Create a blank dictionary
for i,c in enumerate(communities): # Loop through the list of communities, keeping track of the number for the community
    for name in c: # Loop through each person in a community
        modularity_dict[name] = i # Create an entry in the dictionary for the person, where the value is which group they belong to.

# Now you can add modularity information like we did the other metrics
nx.set_node_attributes(G, modularity_dict, 'modularity')

# First get a list of just the nodes in that class
class0 = [n for n in G.nodes() if G.nodes[n]['modularity'] == 0]

# Then create a dictionary of the eigenvector centralities of those nodes
class0_eigenvector = {n:G.nodes[n]['eigenvector'] for n in class0}

# Then sort that dictionary and print the first 5 results
class0_sorted_by_eigenvector = sorted(class0_eigenvector.items(), key=itemgetter(1), reverse=True)

print("Modularity Class 0 Sorted by Eigenvector Centrality:")
for node in class0_sorted_by_eigenvector[:5]:
    print("Name:", node[0], "| Eigenvector Centrality:", node[1])

for i,c in enumerate(communities): # Loop through the list of communities
    if len(c) > 2: # Filter out modularity classes with 2 or fewer nodes
        print('\nClass '+str(i)+':', list(c)) # Print out the classes and their members

nx.write_gexf(G, '{}_network.gexf'.format(country))
print('\n\n\n\nOutput gexf created!!!!')

Network density: 0.004413150195864531
False
Network diameter of largest component: 13
Triadic closure: 0
Top 20 nodes by degree:
('4437233895', 108)
('4224729994', 47)
('357128018', 44)
('704007117695524868', 37)
('4840551713', 34)
('3899481526', 29)
('801463213833547776', 28)
('4272870988', 24)
('635452378', 23)
('108975851', 17)
('299148448', 16)
('1f33d9e4bebd609cdefbf41afeba18c67fece728403a21ae9ff26430748359bd', 16)
('512283231', 16)
('2904103115', 15)
('335288928', 15)
('905141386781085696', 14)
('3011134177', 12)
('4508630900', 12)
('1621793918', 12)
('1220760792', 12)
Top 20 nodes by betweenness centrality:
('4437233895', 0.3892649522399654)
('4224729994', 0.15915911523760964)
('4840551713', 0.13740625146870705)
('357128018', 0.12802850482053568)
('4272870988', 0.10999223137689874)
('3899481526', 0.09012242654099444)
('704007117695524868', 0.08624653431267508)
('299148448', 0.08158538146321803)
('1407756a21b7198310b6576a2bfa912c0ae927104ccef18be3584d1bde47e0d2', 0.06807873600244

Modularity Class 0 Sorted by Eigenvector Centrality:
Name: 4224729994 | Eigenvector Centrality: 0.08033630168048622
Name: pH9xDL2DavEsehO1wXeGFx4fcz0kUuHsLeUzc5wQNo= | Eigenvector Centrality: 0.07118250092868597
Name: 4272870988 | Eigenvector Centrality: 0.0190713934456335
Name: Y7YkxQS9OwrebhLB73mjALKlR5mmyjrlHiPGb+MfsQ= | Eigenvector Centrality: 0.011444386357383379
Name: 3676820373 | Eigenvector Centrality: 0.009930989307171059

Class 0: ['QkIeeNRaiFXxWwo5+Lh69ueyUTjNuZ2CNYQLWlFWDaQ=', 'YLUEtwZUqZ0x+tz0dko5LNT6k4calFXfhG9wnqskA=', 'vuipVTwyUcCmEXQjyn7+z8WH4kkb46Hw80FpAOMETm0=', 'YbiSgOteOqAiqYFtV2meuzfim9nADNzJHI3CtTjgGXc=', 'c2d97b640b39abe793318cef280053e417122fb9bb828a4d32fb5a17c21df637', '4355797094', 'pR8RZniCE5lh00zlro7eTu1LeG0S5QpNtlvK2Q9CQg=', 'ywPlp1oJG5OGJBcDTWrbnp77upOJPMXP9rOQylb36o=', '3676820373', 'PwP+3ciiNBrrASmlVL1Yif3lgP4VOJkrFDZSDmLgQHo=', 'Oj9jbsNh7K8NdhR2oO11y7IKVkgfcEtlkyhMEym+77g=', '6421e03ec457130bf3c6d045fef58e22bfd4977d159c790205e399b9cc8411c1', 'msk4RpZab

In [51]:
G=nx.from_pandas_edgelist(df, 'Source', 'Target', ['Weight'])
connections = [i['Weight'] for i in dict(G.edges).values()]

df_attributes_only['country']=pd.Categorical(df_attributes_only['country'])
df_attributes_only['country'].cat.codes

#plt.legend(loc='center')


fig, ax = plt.subplots(figsize=(100,100))
pos = nx.spring_layout(G)
#nx.draw_networkx_edges(G, pos, width=connections, ax=ax)
nx.draw(G, with_labels=False, 
        node_color=df_attributes_only['codes'], 
        cmap=plt.cm.Set3, node_size=100, width=connections,
        alpha = .70, ax = ax, pos = pos)

plt.draw()
#plt.savefig('Test.png')
plt.show()

NetworkXError: Invalid edge_attr argument: ['Weight']

In [35]:
node_attr

{'+AL7riKucmp7iF1r1UxC5KytaCNzV1Y+ykf83lDdw=': {'country': 'iran'},
 '0M6q78TgoPkgXsQlLdCkIbNgW1tTqbN6Xe1234IaQ=': {'country': 'iran'},
 '2c538df357dec4c1413851b5f1b68da2d7d1f6df4157be85d15901b0913ff714': {'country': 'iran'},
 '5eb2c59301da5d5249e7c40e1c4abe22dfd9e9e2a3b6ce60fa9bb080b673e61c': {'country': 'iran'},
 '719e0500827a4fdcf610520054f04d4ec13c7125b83c0a23fc6c75ca86a8cf6b': {'country': 'iran'},
 '7AihqeEJRHKt30Sl81r6lQenHQc8IR1yGv6Yc+yzTU=': {'country': 'iran'},
 '7H7hry65IojQY4jTQFoUPhBu6voB6Qz6lJ38+oNHOrE=': {'country': 'iran'},
 '7tvS6RtKDHvcHubX4qpH0gwH0DFwVpBhMFu9EwkyBM=': {'country': 'iran'},
 '95RWp7S3oRpL0XfKu3kymI0Li9Q74BukxvgbpdW0Gg=': {'country': 'iran'},
 '9PZgPUKeb14sqtKQg0Gv1RQj2+BDAcgbytJ6Q7f8f54=': {'country': 'iran'},
 'Abwu6+0+JS0EiagVTAOQMIzYJbxbR3tnRJ5Y3qjR8c=': {'country': 'iran'},
 'aGj8ZHdbONkeiIdk5MZ10Llz3JqjsuBbZHfAp8riMyU=': {'country': 'iran'},
 'Au2wwfJNPHStbKAqBJlcFz3RfmhCjiUuXnEyFtrdUL4=': {'country': 'iran'},
 'bgUEpUr07MmsAhTLcaWA17YIyOW3msbYARWN